In [1]:
%pip install pyspark 

Note: you may need to restart the kernel to use updated packages.


###  Préparation de l'environnement

In [2]:
import sys
import pyspark
from pathlib import Path

projet = Path.home() / "NoSQL, transformations et Data Lakehouse"
dossier_data = projet / "data"
dossier_output = projet / "output"
dossier_data.mkdir(parents=True, exist_ok=True)
dossier_output.mkdir(parents=True, exist_ok=True)
print("=== ENVIRONNEMENT TP3 ===")
print("Version Python :", sys.version.split()[0])
print("Version PySpark :", pyspark.__version__)
print("Mode d'exécution :", "Local")
print("Dossier du projet :", projet)
print("Données sources :", dossier_data)
print("Résultats :", dossier_output)
import os
print("Processeurs disponibles :", os.cpu_count())

=== ENVIRONNEMENT TP3 ===
Version Python : 3.13.9
Version PySpark : 4.3.0
Mode d'exécution : Local
Dossier du projet : C:\Users\houda\NoSQL, transformations et Data Lakehouse
Données sources : C:\Users\houda\NoSQL, transformations et Data Lakehouse\data
Résultats : C:\Users\houda\NoSQL, transformations et Data Lakehouse\output
Processeurs disponibles : 8


### Vérification de l'écriture d'un fichier de test

In [3]:
import csv
chemin_test = dossier_output / "test_ecriture.csv"
donnees = [
    ["magasin", "chiffre_affaires"],
    ["Magasin A", 1200],
    ["Magasin B", 850],
    ["Magasin C", 1600]
]
with open(chemin_test, "w", newline="", encoding="utf-8") as fichier:
    writer = csv.writer(fichier)
    writer.writerows(donnees)
print("Fichier créé :", chemin_test)
print("Fichier présent :", chemin_test.exists())

Fichier créé : C:\Users\houda\NoSQL, transformations et Data Lakehouse\output\test_ecriture.csv
Fichier présent : True


## Étape 2 : Inventaire des sources disponibles

### Identification des fichiers sources du projet

In [4]:
import zipfile
from pathlib import Path
chemin_zip = Path.home() / "Downloads" / "retail_project.zip"
with zipfile.ZipFile(chemin_zip, "r") as archive:
    fichiers = archive.namelist()

    print("Nombre d'éléments :", len(fichiers))
    print("\nContenu de l'archive :")

    for fichier in fichiers:
        print(fichier)

Nombre d'éléments : 73

Contenu de l'archive :
retail_project/
retail_project/data/
retail_project/data/customer/
retail_project/data/customer/customer.dat
retail_project/data/customer_address/
retail_project/data/customer_address/customer_address.dat
retail_project/data/customer_demographics/
retail_project/data/customer_demographics/customer_demographics.dat
retail_project/data/date_dim/
retail_project/data/date_dim/date_dim.dat
retail_project/data/household_demographics/
retail_project/data/household_demographics/household_demographics.dat
retail_project/data/income_band/
retail_project/data/income_band/income_band.dat
retail_project/data/inventory/
retail_project/data/inventory/inventory.dat
retail_project/data/item/
retail_project/data/item/item.dat
retail_project/data/item_marketprices/
retail_project/data/item_marketprices/item_marketprices.dat
retail_project/data/item_marketprices/item_marketprices_audit.csv
retail_project/data/product_reviews/
retail_project/data/product_revie

In [5]:
import os
import sys

from pyspark.sql import SparkSession

java_home = r"C:\Program Files\Eclipse Adoptium\jdk-21.0.13.9-hotspot"

os.environ["JAVA_HOME"] = java_home

os.environ["PATH"] = (
    os.path.join(java_home, "bin")
    + os.pathsep
    + os.path.dirname(sys.executable)
    + os.pathsep
    + os.environ["PATH"]
)


os.environ["PYSPARK_PYTHON"] = sys.executable
os.environ["PYSPARK_DRIVER_PYTHON"] = sys.executable


spark = (
    SparkSession.builder
    .appName("TP3-Retail-Intelligence")
    .master("local[*]")
    .getOrCreate()
)

print("Session Spark initialisée.")
print("Version Spark :", spark.version)

Session Spark initialisée.
Version Spark : 4.3.0


### Sélection des sources : ventes et articles

In [6]:
import zipfile
sources = [
    "retail_project/data/store_sales/store_sales.dat",
    "retail_project/data/item/item.dat"
]
with zipfile.ZipFile(chemin_zip, "r") as archive:
    for source in sources:
        archive.extract(source, path=dossier_data)

        print("Source extraite :", source)

print("\nExtraction terminée.")

Source extraite : retail_project/data/store_sales/store_sales.dat
Source extraite : retail_project/data/item/item.dat

Extraction terminée.


### Inventaire technique des deux sources sélectionnées

In [7]:

fichiers = {
    "Ventes magasins": dossier_data / "retail_project/data/store_sales/store_sales.dat",
    "Articles": dossier_data / "retail_project/data/item/item.dat"
}


for nom, chemin in fichiers.items():
    taille_mo = chemin.stat().st_size / (1024 * 1024)

    print("Source :", nom)
    print("Nom du fichier :", chemin.name)
    print("Format :", chemin.suffix)
    print("Taille :", round(taille_mo, 2), "Mo")
    print("Chemin :", chemin)
    print("-" * 50)

Source : Ventes magasins
Nom du fichier : store_sales.dat
Format : .dat
Taille : 88.47 Mo
Chemin : C:\Users\houda\NoSQL, transformations et Data Lakehouse\data\retail_project\data\store_sales\store_sales.dat
--------------------------------------------------
Source : Articles
Nom du fichier : item.dat
Format : .dat
Taille : 6.48 Mo
Chemin : C:\Users\houda\NoSQL, transformations et Data Lakehouse\data\retail_project\data\item\item.dat
--------------------------------------------------


### Identification de la structure des fichiers sources

In [8]:
for nom, chemin in fichiers.items():

    print("\nSource :", nom)

    # Ouvrir le fichier en lecture
    with open(chemin, "r", encoding="utf-8", errors="replace") as f:

        # Lire les trois premières lignes
        for i in range(3):
            ligne = f.readline()
            print(f"Ligne {i + 1} :", repr(ligne[:250]))

    print("-" * 50)


Source : Ventes magasins
Ligne 1 : '36890|71480|10001|44404|493944|2898|44093|10|294|0|31|18.00|30.78|23.08|238.70|715.48|558.00|954.18|21.46|0.00|715.48|736.94|157.48\n'
Ligne 2 : '36890|71480|10000|44404|493944|2898|44093|10|279|0|54|99.21|148.82|83.34|3535.92|4500.36|5357.34|8036.28|225.02|0.00|4500.36|4725.38|-856.98\n'
Ligne 3 : '36890|71480|15556|44404|493944|2898|44093|10|114|0|39|79.49|94.59|83.24|442.65|3246.36|3100.11|3689.01|184.07|616.81|2629.55|2813.62|-470.56\n'
--------------------------------------------------

Source : Articles
Ligne 1 : '0|AAAAAAAAAAAAAAAA|2000-01-14||quickly even dinos beneath the frays must have to boost boldly careful bold escapades: stealthily even forges over the dependencies integrate always past the quiet sly decoys-- notornis sol|72.29|64.96|3898712|71TS7NSbv'
Ligne 2 : '1|AAAAAAAAAAAAAAAB|2000-09-23||quietly quiet hockey players outside the furiously fluffy sauternes impress daringly silently quiet gifts? bold ruthless escapades run slyly ep

### Comptage des lignes et des colonnes des sources

In [9]:
for nom, chemin in fichiers.items():

    # Compter les lignes
    with open(chemin, "r", encoding="utf-8", errors="replace") as f:
        nombre_lignes = sum(1 for ligne in f)

    # Compter les colonnes à partir de la première ligne
    with open(chemin, "r", encoding="utf-8", errors="replace") as f:
        premiere_ligne = f.readline().rstrip("\r\n")
        nombre_colonnes = len(premiere_ligne.split("|"))

    print("Source :", nom)
    print("Nombre de lignes :", nombre_lignes)
    print("Nombre de colonnes :", nombre_colonnes)
    print("-" * 40)

Source : Ventes magasins
Nombre de lignes : 667579
Nombre de colonnes : 23
----------------------------------------
Source : Articles
Nombre de lignes : 17820
Nombre de colonnes : 22
----------------------------------------


### Identification des types de données des sources

In [10]:
import pandas as pd
for nom, chemin in fichiers.items():

    # Lire un échantillon de 1000 lignes
    df = pd.read_csv(
        chemin,
        sep="|",
        header=None,
        nrows=1000,
        low_memory=False
    )

    print("\nSource :", nom)
    print("Colonnes et types détectés :")

    for colonne, type_donnee in df.dtypes.items():
        print(f"Colonne {colonne} : {type_donnee}")

    print("-" * 40)


Source : Ventes magasins
Colonnes et types détectés :
Colonne 0 : int64
Colonne 1 : int64
Colonne 2 : int64
Colonne 3 : int64
Colonne 4 : int64
Colonne 5 : int64
Colonne 6 : int64
Colonne 7 : int64
Colonne 8 : int64
Colonne 9 : int64
Colonne 10 : int64
Colonne 11 : float64
Colonne 12 : float64
Colonne 13 : float64
Colonne 14 : float64
Colonne 15 : float64
Colonne 16 : float64
Colonne 17 : float64
Colonne 18 : float64
Colonne 19 : float64
Colonne 20 : float64
Colonne 21 : float64
Colonne 22 : float64
----------------------------------------

Source : Articles
Colonnes et types détectés :
Colonne 0 : int64
Colonne 1 : object
Colonne 2 : object
Colonne 3 : float64
Colonne 4 : object
Colonne 5 : float64
Colonne 6 : float64
Colonne 7 : int64
Colonne 8 : object
Colonne 9 : int64
Colonne 10 : object
Colonne 11 : int64
Colonne 12 : object
Colonne 13 : int64
Colonne 14 : object
Colonne 15 : object
Colonne 16 : object
Colonne 17 : object
Colonne 18 : object
Colonne 19 : object
Colonne 20 : int6

### Identification des clés possibles et des premières anomalies

In [11]:
import pandas as pd
for nom, chemin in fichiers.items():

    # Lecture des données
    df = pd.read_csv(
        chemin,
        sep="|",
        header=None
    )

    print("\nSource :", nom)

    # Vérification de la première colonne
    print("Nombre de lignes :", len(df))
    print("Valeurs uniques colonne 0 :", df[0].nunique())
    print("Doublons colonne 0 :", df[0].duplicated().sum())

    # Recherche des valeurs manquantes
    print("Valeurs manquantes :", df.isnull().sum().sum())

    print("-" * 40)


Source : Ventes magasins
Nombre de lignes : 667579
Valeurs uniques colonne 0 : 1808
Doublons colonne 0 : 665771
Valeurs manquantes : 1139
----------------------------------------

Source : Articles
Nombre de lignes : 17820
Valeurs uniques colonne 0 : 17820
Doublons colonne 0 : 0
Valeurs manquantes : 21203
----------------------------------------


### Vérification de la structure des fichiers sources

In [12]:
for nom, chemin in fichiers.items():

    nombre_colonnes_attendu = 23 if nom == "Ventes magasins" else 22
    lignes_incorrectes = 0

    with open(chemin, "r", encoding="utf-8", errors="replace") as f:
        for ligne in f:
            colonnes = ligne.rstrip("\r\n").split("|")

            if len(colonnes) != nombre_colonnes_attendu:
                lignes_incorrectes += 1

    print("Source :", nom)
    print("Colonnes attendues :", nombre_colonnes_attendu)
    print("Lignes mal structurées :", lignes_incorrectes)
    print("-" * 40)

Source : Ventes magasins
Colonnes attendues : 23
Lignes mal structurées : 0
----------------------------------------
Source : Articles
Colonnes attendues : 22
Lignes mal structurées : 0
----------------------------------------


## Étape 3 : Lecture et exploration des données avec PySpark



### Chargement des deux sources dans des DataFrames Spark

In [13]:

chemin_ventes = fichiers["Ventes magasins"]
chemin_articles = fichiers["Articles"]
df_ventes = (
    spark.read
    .option("sep", "|")
    .option("header", "false")
    .csv(str(chemin_ventes))
)

df_articles = (
    spark.read
    .option("sep", "|")
    .option("header", "false")
    .csv(str(chemin_articles))
)

print("Chargement des deux sources réussi.")
print("Colonnes des ventes :", len(df_ventes.columns))
print("Colonnes des articles :", len(df_articles.columns))

Chargement des deux sources réussi.
Colonnes des ventes : 23
Colonnes des articles : 22


### Affichage des schémas des DataFrames Spark

In [14]:
print("=== SCHÉMA DES VENTES ===")
df_ventes.printSchema()
print("=== SCHÉMA DES ARTICLES ===")
df_articles.printSchema()

=== SCHÉMA DES VENTES ===
root
 |-- _c0: string (nullable = true)
 |-- _c1: string (nullable = true)
 |-- _c2: string (nullable = true)
 |-- _c3: string (nullable = true)
 |-- _c4: string (nullable = true)
 |-- _c5: string (nullable = true)
 |-- _c6: string (nullable = true)
 |-- _c7: string (nullable = true)
 |-- _c8: string (nullable = true)
 |-- _c9: string (nullable = true)
 |-- _c10: string (nullable = true)
 |-- _c11: string (nullable = true)
 |-- _c12: string (nullable = true)
 |-- _c13: string (nullable = true)
 |-- _c14: string (nullable = true)
 |-- _c15: string (nullable = true)
 |-- _c16: string (nullable = true)
 |-- _c17: string (nullable = true)
 |-- _c18: string (nullable = true)
 |-- _c19: string (nullable = true)
 |-- _c20: string (nullable = true)
 |-- _c21: string (nullable = true)
 |-- _c22: string (nullable = true)

=== SCHÉMA DES ARTICLES ===
root
 |-- _c0: string (nullable = true)
 |-- _c1: string (nullable = true)
 |-- _c2: string (nullable = true)
 |-- _c3: st

### Visualisation des premières lignes des sources

In [15]:
# Afficher les 5 premières lignes des ventes
print("=== VENTES MAGASINS ===")
df_ventes.show(5, truncate=False)

# Afficher les 5 premières lignes des articles
print("=== ARTICLES ===")
df_articles.show(5, truncate=False)

=== VENTES MAGASINS ===
+-----+-----+-----+-----+-------+----+-----+---+---+---+----+-----+------+------+-------+-------+-------+-------+------+------+-------+-------+-------+
|_c0  |_c1  |_c2  |_c3  |_c4    |_c5 |_c6  |_c7|_c8|_c9|_c10|_c11 |_c12  |_c13  |_c14   |_c15   |_c16   |_c17   |_c18  |_c19  |_c20   |_c21   |_c22   |
+-----+-----+-----+-----+-------+----+-----+---+---+---+----+-----+------+------+-------+-------+-------+-------+------+------+-------+-------+-------+
|36890|71480|10001|44404|493944 |2898|44093|10 |294|0  |31  |18.00|30.78 |23.08 |238.70 |715.48 |558.00 |954.18 |21.46 |0.00  |715.48 |736.94 |157.48 |
|36890|71480|10000|44404|493944 |2898|44093|10 |279|0  |54  |99.21|148.82|83.34 |3535.92|4500.36|5357.34|8036.28|225.02|0.00  |4500.36|4725.38|-856.98|
|36890|71480|15556|44404|493944 |2898|44093|10 |114|0  |39  |79.49|94.59 |83.24 |442.65 |3246.36|3100.11|3689.01|184.07|616.81|2629.55|2813.62|-470.56|
|36890|45392|4597 |89832|1774847|6266|16285|4  |285|1  |20  |74.

### Vérification du nombre d'enregistrements avec PySpark

In [16]:
# Compter les enregistrements avec Spark

nombre_ventes = df_ventes.count()
nombre_articles = df_articles.count()

print("Nombre de ventes :", nombre_ventes)
print("Nombre d'articles :", nombre_articles)

Nombre de ventes : 667579
Nombre d'articles : 17820


### Contrôle des valeurs manquantes avec PySpark

In [17]:
from pyspark.sql import functions as F
for nom, df in [
    ("Ventes magasins", df_ventes),
    ("Articles", df_articles)
]:
    print("\n=== ", nom, " ===")

    # Compter les valeurs nulles de chaque colonne
    resultats = df.select([
        F.count(F.when(F.col(colonne).isNull(), 1)).alias(colonne)
        for colonne in df.columns
    ]).first()

    # Afficher uniquement les colonnes contenant des valeurs nulles
    for colonne in df.columns:
        nombre = resultats[colonne]

        if nombre > 0:
            print(colonne, ":", nombre, "valeurs manquantes")

    print("Total :", sum(resultats))


===  Ventes magasins  ===
_c4 : 238 valeurs manquantes
_c5 : 295 valeurs manquantes
_c6 : 448 valeurs manquantes
_c8 : 158 valeurs manquantes
Total : 1139

===  Articles  ===
_c3 : 17820 valeurs manquantes
_c4 : 4 valeurs manquantes
_c7 : 8 valeurs manquantes
_c8 : 6 valeurs manquantes
Total : 17838


### Vérification des valeurs manquantes représentées par du texte

In [18]:
from pyspark.sql import functions as F

# Valeurs textuelles pouvant représenter une donnée manquante
valeurs_a_verifier = ["", "N/A", "NA", "NULL", "None", "NaN"]

print("=== ARTICLES : VALEURS MANQUANTES TEXTUELLES ===")

for valeur in valeurs_a_verifier:

    total = df_articles.select([
        F.sum(
            F.when(F.col(colonne) == valeur, 1).otherwise(0)
        ).alias(colonne)
        for colonne in df_articles.columns
    ]).first()

    nombre = sum(total)

    if nombre > 0:
        print(repr(valeur), ":", nombre)

=== ARTICLES : VALEURS MANQUANTES TEXTUELLES ===
'N/A' : 3365


### Détection des doublons avec PySpark

In [19]:
for nom, df in [
    ("Ventes magasins", df_ventes),
    ("Articles", df_articles)
]:

    nombre_total = df.count()
    nombre_uniques = df.dropDuplicates().count()
    nombre_doublons = nombre_total - nombre_uniques

    print("\nSource :", nom)
    print("Nombre total de lignes :", nombre_total)
    print("Nombre de lignes distinctes :", nombre_uniques)
    print("Nombre de doublons :", nombre_doublons)


Source : Ventes magasins
Nombre total de lignes : 667579
Nombre de lignes distinctes : 667579
Nombre de doublons : 0

Source : Articles
Nombre total de lignes : 17820
Nombre de lignes distinctes : 17820
Nombre de doublons : 0


### Recherche de valeurs atypiques dans les ventes

In [20]:
from pyspark.sql import functions as F
df_controle = df_ventes.select(
    F.col("_c10").cast("int").alias("colonne_10"),
    F.col("_c11").cast("double").alias("colonne_11")
)

# Calculer les valeurs minimales et maximales
df_controle.select(
    F.min("colonne_10").alias("minimum_colonne_10"),
    F.max("colonne_10").alias("maximum_colonne_10"),
    F.min("colonne_11").alias("minimum_colonne_11"),
    F.max("colonne_11").alias("maximum_colonne_11")
).show()

+------------------+------------------+------------------+------------------+
|minimum_colonne_10|maximum_colonne_10|minimum_colonne_11|maximum_colonne_11|
+------------------+------------------+------------------+------------------+
|                 1|               100|               1.0|             100.0|
+------------------+------------------+------------------+------------------+



### Identification des noms métiers des colonnes TPC-DS

In [21]:
# Noms des colonnes de la table des ventes (TPC-DS)
colonnes_ventes = [
    "ss_sold_date_sk",
    "ss_sold_time_sk",
    "ss_item_sk",
    "ss_customer_sk",
    "ss_cdemo_sk",
    "ss_hdemo_sk",
    "ss_addr_sk",
    "ss_store_sk",
    "ss_promo_sk",
    "ss_ticket_number",
    "ss_quantity",
    "ss_wholesale_cost",
    "ss_list_price",
    "ss_sales_price",
    "ss_ext_discount_amt",
    "ss_ext_sales_price",
    "ss_ext_wholesale_cost",
    "ss_ext_list_price",
    "ss_ext_tax",
    "ss_coupon_amt",
    "ss_net_paid",
    "ss_net_paid_inc_tax",
    "ss_net_profit"
]

# Noms des colonnes de la table des articles (TPC-DS)
colonnes_articles = [
    "i_item_sk",
    "i_item_id",
    "i_rec_start_date",
    "i_rec_end_date",
    "i_item_desc",
    "i_current_price",
    "i_wholesale_cost",
    "i_brand_id",
    "i_brand",
    "i_class_id",
    "i_class",
    "i_category_id",
    "i_category",
    "i_manufact_id",
    "i_manufact",
    "i_size",
    "i_formulation",
    "i_color",
    "i_units",
    "i_container",
    "i_manager_id",
    "i_product_name"
]

# Vérifier que le nombre de noms correspond aux colonnes des fichiers
print("=== VÉRIFICATION DES SCHÉMAS ===")

print("Ventes :")
print("Colonnes du fichier :", len(df_ventes.columns))
print("Noms métiers disponibles :", len(colonnes_ventes))

print("\nArticles :")
print("Colonnes du fichier :", len(df_articles.columns))
print("Noms métiers disponibles :", len(colonnes_articles))

=== VÉRIFICATION DES SCHÉMAS ===
Ventes :
Colonnes du fichier : 23
Noms métiers disponibles : 23

Articles :
Colonnes du fichier : 22
Noms métiers disponibles : 22


### Vérification de la correspondance entre les identifiants des articles

In [22]:
from pyspark.sql import functions as F

# Identifiants d'articles présents dans les ventes
articles_ventes = (
    df_ventes
    .select(F.col("_c2").alias("item_id"))
    .where(F.col("item_id").isNotNull())
    .distinct()
)

# Identifiants d'articles présents dans le catalogue
articles_catalogue = (
    df_articles
    .select(F.col("_c0").alias("item_id"))
    .distinct()
)

# Vérifier la correspondance des identifiants
nombre_articles_ventes = articles_ventes.count()

nombre_articles_correspondants = (
    articles_ventes
    .join(articles_catalogue, on="item_id", how="inner")
    .count()
)

nombre_articles_sans_correspondance = (
    articles_ventes
    .join(articles_catalogue, on="item_id", how="left_anti")
    .count()
)

print("Articles distincts dans les ventes :", nombre_articles_ventes)
print("Articles retrouvés dans le catalogue :", nombre_articles_correspondants)
print("Articles sans correspondance :", nombre_articles_sans_correspondance)

Articles distincts dans les ventes : 17820
Articles retrouvés dans le catalogue : 17820
Articles sans correspondance : 0


### Renommage des colonnes avec les noms métiers TPC-DS

In [23]:
# Renommer les colonnes des ventes
df_ventes_nom = df_ventes.toDF(*colonnes_ventes)

# Renommer les colonnes des articles
df_articles_nom = df_articles.toDF(*colonnes_articles)

print("=== COLONNES DES VENTES ===")
print(df_ventes_nom.columns)
print("\n=== COLONNES DES ARTICLES ===")
print(df_articles_nom.columns)

=== COLONNES DES VENTES ===
['ss_sold_date_sk', 'ss_sold_time_sk', 'ss_item_sk', 'ss_customer_sk', 'ss_cdemo_sk', 'ss_hdemo_sk', 'ss_addr_sk', 'ss_store_sk', 'ss_promo_sk', 'ss_ticket_number', 'ss_quantity', 'ss_wholesale_cost', 'ss_list_price', 'ss_sales_price', 'ss_ext_discount_amt', 'ss_ext_sales_price', 'ss_ext_wholesale_cost', 'ss_ext_list_price', 'ss_ext_tax', 'ss_coupon_amt', 'ss_net_paid', 'ss_net_paid_inc_tax', 'ss_net_profit']

=== COLONNES DES ARTICLES ===
['i_item_sk', 'i_item_id', 'i_rec_start_date', 'i_rec_end_date', 'i_item_desc', 'i_current_price', 'i_wholesale_cost', 'i_brand_id', 'i_brand', 'i_class_id', 'i_class', 'i_category_id', 'i_category', 'i_manufact_id', 'i_manufact', 'i_size', 'i_formulation', 'i_color', 'i_units', 'i_container', 'i_manager_id', 'i_product_name']


### Conversion des types de données des ventes

In [24]:
from pyspark.sql import functions as F

# Colonnes contenant des identifiants ou des quantités entières
colonnes_entieres = colonnes_ventes[:11]

# Colonnes contenant des montants décimaux
colonnes_decimales = colonnes_ventes[11:]

# Créer un nouveau DataFrame pour les données transformées
df_ventes_type = df_ventes_nom

# Convertir les colonnes entières
for colonne in colonnes_entieres:
    df_ventes_type = df_ventes_type.withColumn(
        colonne,
        F.col(colonne).cast("int")
    )

# Convertir les montants décimaux
for colonne in colonnes_decimales:
    df_ventes_type = df_ventes_type.withColumn(
        colonne,
        F.col(colonne).cast("decimal(18,2)")
    )

# Vérifier les nouveaux types
df_ventes_type.printSchema()

root
 |-- ss_sold_date_sk: integer (nullable = true)
 |-- ss_sold_time_sk: integer (nullable = true)
 |-- ss_item_sk: integer (nullable = true)
 |-- ss_customer_sk: integer (nullable = true)
 |-- ss_cdemo_sk: integer (nullable = true)
 |-- ss_hdemo_sk: integer (nullable = true)
 |-- ss_addr_sk: integer (nullable = true)
 |-- ss_store_sk: integer (nullable = true)
 |-- ss_promo_sk: integer (nullable = true)
 |-- ss_ticket_number: integer (nullable = true)
 |-- ss_quantity: integer (nullable = true)
 |-- ss_wholesale_cost: decimal(18,2) (nullable = true)
 |-- ss_list_price: decimal(18,2) (nullable = true)
 |-- ss_sales_price: decimal(18,2) (nullable = true)
 |-- ss_ext_discount_amt: decimal(18,2) (nullable = true)
 |-- ss_ext_sales_price: decimal(18,2) (nullable = true)
 |-- ss_ext_wholesale_cost: decimal(18,2) (nullable = true)
 |-- ss_ext_list_price: decimal(18,2) (nullable = true)
 |-- ss_ext_tax: decimal(18,2) (nullable = true)
 |-- ss_coupon_amt: decimal(18,2) (nullable = true)
 |--

### Conversion des types de données des articles

In [25]:
from pyspark.sql import functions as F

# Créer un DataFrame pour les articles transformés
df_articles_type = df_articles_nom
colonnes_entieres_articles = [
    "i_item_sk",
    "i_brand_id",
    "i_class_id",
    "i_category_id",
    "i_manufact_id",
    "i_manager_id"
]

colonnes_decimales_articles = [
    "i_current_price",
    "i_wholesale_cost"
]

# Colonnes contenant des dates
colonnes_dates_articles = [
    "i_rec_start_date",
    "i_rec_end_date"
]


for colonne in colonnes_entieres_articles:
    df_articles_type = df_articles_type.withColumn(
        colonne,
        F.col(colonne).cast("int")
    )


for colonne in colonnes_decimales_articles:
    df_articles_type = df_articles_type.withColumn(
        colonne,
        F.col(colonne).cast("decimal(18,2)")
    )


for colonne in colonnes_dates_articles:
    df_articles_type = df_articles_type.withColumn(
        colonne,
        F.to_date(F.col(colonne), "yyyy-MM-dd")
    )

# Affichage du schéma
df_articles_type.printSchema()

root
 |-- i_item_sk: integer (nullable = true)
 |-- i_item_id: string (nullable = true)
 |-- i_rec_start_date: date (nullable = true)
 |-- i_rec_end_date: date (nullable = true)
 |-- i_item_desc: string (nullable = true)
 |-- i_current_price: decimal(18,2) (nullable = true)
 |-- i_wholesale_cost: decimal(18,2) (nullable = true)
 |-- i_brand_id: integer (nullable = true)
 |-- i_brand: string (nullable = true)
 |-- i_class_id: integer (nullable = true)
 |-- i_class: string (nullable = true)
 |-- i_category_id: integer (nullable = true)
 |-- i_category: string (nullable = true)
 |-- i_manufact_id: integer (nullable = true)
 |-- i_manufact: string (nullable = true)
 |-- i_size: string (nullable = true)
 |-- i_formulation: string (nullable = true)
 |-- i_color: string (nullable = true)
 |-- i_units: string (nullable = true)
 |-- i_container: string (nullable = true)
 |-- i_manager_id: integer (nullable = true)
 |-- i_product_name: string (nullable = true)



### Vérification des valeurs nulles après conversion des types

In [26]:
from pyspark.sql import functions as F

# Comparer les valeurs nulles avant et après conversion
for nom, avant, apres in [
    ("Ventes magasins", df_ventes_nom, df_ventes_type),
    ("Articles", df_articles_nom, df_articles_type)
]:

    nulls_avant = avant.select([
        F.sum(F.col(c).isNull().cast("int")).alias(c)
        for c in avant.columns
    ]).first()

    nulls_apres = apres.select([
        F.sum(F.col(c).isNull().cast("int")).alias(c)
        for c in apres.columns
    ]).first()

    print("\n=== ", nom, " ===")

    for colonne in avant.columns:
        difference = nulls_apres[colonne] - nulls_avant[colonne]

        if difference > 0:
            print(colonne, ":", difference, "nouveaux NULL")

    print("Total avant :", sum(nulls_avant))
    print("Total après :", sum(nulls_apres))


===  Ventes magasins  ===
Total avant : 1139
Total après : 1139

===  Articles  ===
Total avant : 17838
Total après : 17838


### Normalisation des valeurs manquantes du catalogue

In [27]:
from pyspark.sql import functions as F

df_articles_clean = df_articles_type
# Remplacer les textes "N/A" par des valeurs nulles
df_articles_clean = df_articles_clean.replace("N/A", None)
# Vérifier le nombre total de valeurs nulles
resultat = df_articles_clean.select([
    F.count(F.when(F.col(c).isNull(), 1)).alias(c)
    for c in df_articles_clean.columns
]).first()

print("Total des valeurs manquantes :", sum(resultat))

Total des valeurs manquantes : 21203


### Vérification des champs essentiels des ventes

In [28]:
from pyspark.sql import functions as F

# Vérifier les champs essentiels des ventes
df_ventes_type.select(
    F.count("*").alias("total_ventes"),
    F.sum(
        F.col("ss_item_sk").isNull().cast("int")
    ).alias("articles_manquants"),
    F.sum(
        F.col("ss_quantity").isNull().cast("int")
    ).alias("quantites_manquantes"),
    F.sum(
        (F.col("ss_quantity") <= 0).cast("int")
    ).alias("quantites_non_positives")
).show()

+------------+------------------+--------------------+-----------------------+
|total_ventes|articles_manquants|quantites_manquantes|quantites_non_positives|
+------------+------------------+--------------------+-----------------------+
|      667579|                 0|                   0|                      0|
+------------+------------------+--------------------+-----------------------+



### Création du DataFrame des ventes nettoyées

In [29]:
df_ventes_clean = df_ventes_type.filter(
    (F.col("ss_item_sk").isNotNull()) &
    (F.col("ss_quantity") > 0))

print("Ventes initiales :", df_ventes_type.count())
print("Ventes conservées :", df_ventes_clean.count())
print("Ventes exclues :", df_ventes_type.count() - df_ventes_clean.count())

Ventes initiales : 667579
Ventes conservées : 667579
Ventes exclues : 0


### Vérification des identifiants et des prix des articles

In [30]:
from pyspark.sql import functions as F

df_articles_clean.select(
    F.count("*").alias("total_articles"),

    F.countDistinct("i_item_sk").alias("identifiants_uniques"),

    F.sum(
        F.col("i_item_sk").isNull().cast("int")
    ).alias("identifiants_manquants"),

    F.sum(
        F.col("i_current_price").isNull().cast("int")
    ).alias("prix_manquants"),

    F.sum(
        (F.col("i_current_price") <= 0).cast("int")
    ).alias("prix_non_positifs")
).show()

+--------------+--------------------+----------------------+--------------+-----------------+
|total_articles|identifiants_uniques|identifiants_manquants|prix_manquants|prix_non_positifs|
+--------------+--------------------+----------------------+--------------+-----------------+
|         17820|               17820|                     0|             0|                0|
+--------------+--------------------+----------------------+--------------+-----------------+



### Jointure entre les ventes et le catalogue des articles

In [31]:
articles_selection = df_articles_clean.select("i_item_sk","i_product_name","i_category","i_current_price")

df_ventes_enrichies = df_ventes_clean.join(articles_selection,df_ventes_clean["ss_item_sk"] == articles_selection["i_item_sk"],"left")

df_ventes_enrichies = df_ventes_enrichies.drop("i_item_sk")

print("Ventes initiales :", df_ventes_clean.count())
print("Ventes après jointure :", df_ventes_enrichies.count())

df_ventes_enrichies.select(
    "ss_item_sk",
    "ss_quantity",
    "ss_net_paid",
    "i_product_name",
    "i_category",
    "i_current_price").show(5, truncate=25)

Ventes initiales : 667579
Ventes après jointure : 667579
+----------+-----------+-----------+-------------------------+-----------------+---------------+
|ss_item_sk|ss_quantity|ss_net_paid|           i_product_name|       i_category|i_current_price|
+----------+-----------+-----------+-------------------------+-----------------+---------------+
|     10001|         31|     715.48|          6VxbaI6thqY49zO|            Music|           5.98|
|     10000|         54|    4500.36|vwPR6l3UPyCIv5bljZbIG9...|   Home & Kitchen|          70.51|
|     15556|         39|    2629.55|   MSLchXRPaMvAeCqsFrcAJd|            Books|          92.21|
|      4597|         20|    2668.00|Bomxyb5BunmadQEDd8uNT3...|Sports & Outdoors|          56.02|
|      3258|         76|     900.60|CdYnCCNWTmaFy2WRYwFhvP...|            Music|          62.66|
+----------+-----------+-----------+-------------------------+-----------------+---------------+
only showing top 5 rows


### Agrégation des ventes par catégorie de produits

In [32]:
from pyspark.sql import functions as F

# Regrouper les ventes par catégorie
df_ventes_categories = (
    df_ventes_enrichies
    .groupBy("i_category")
    .agg(
        F.count("*").alias("nombre_lignes_vente"),
        F.sum("ss_quantity").alias("quantite_totale"),
        F.sum("ss_net_paid").alias("montant_total")
    )
    .orderBy(F.desc("montant_total"))
)
df_ventes_categories.show(truncate=False)

+------------------------+-------------------+---------------+-------------+
|i_category              |nombre_lignes_vente|quantite_totale|montant_total|
+------------------------+-------------------+---------------+-------------+
|Music                   |301236             |15238374       |519666693.84 |
|Books                   |177671             |9007511        |307151503.25 |
|Movies & TV             |43376              |2189279        |74735891.89  |
|Clothing & Accessories  |32372              |1640550        |56754855.82  |
|Electronics             |32619              |1632098        |55715087.49  |
|Home & Kitchen          |24380              |1226839        |41260511.11  |
|Sports & Outdoors       |23911              |1209305        |40926938.33  |
|Toys & Games            |16121              |809999         |27437825.33  |
|Tools & Home Improvement|15893              |801519         |26880279.31  |
+------------------------+-------------------+---------------+-------------+

## Étape 6 : Production des données Silver



### Enregistrement du catalogue des articles nettoyés

In [33]:
from pyspark.sql import functions as F
dossier_silver_articles = projet / "silver" / "products"
dossier_silver_articles.mkdir(parents=True, exist_ok=True)
df_articles_silver = df_articles_clean.withColumn(
    "source_file",
    F.lit("item.dat")
)

articles_pandas = df_articles_silver.toPandas()
chemin_silver_articles = dossier_silver_articles / "products.parquet"

articles_pandas.to_parquet(
    chemin_silver_articles,
    engine="pyarrow",
    index=False
)

print("Fichier créé :", chemin_silver_articles.exists())
print("Nombre de lignes :", len(articles_pandas))
print("Nombre de colonnes :", len(articles_pandas.columns))
print("Format :", chemin_silver_articles.suffix)

Fichier créé : True
Nombre de lignes : 17820
Nombre de colonnes : 23
Format : .parquet


### Vérification de la lecture du fichier Parquet Silver

In [34]:
import pandas as pd
articles_verification = pd.read_parquet(
    chemin_silver_articles,
    engine="pyarrow")
print("Nombre de lignes :", len(articles_verification))
print("Nombre de colonnes :", len(articles_verification.columns))
print(articles_verification[["i_item_sk", "i_category", "i_current_price", "source_file"]].head())

Nombre de lignes : 17820
Nombre de colonnes : 23
   i_item_sk         i_category i_current_price source_file
0          0  Sports & Outdoors           72.29    item.dat
1          1              Books           95.67    item.dat
2          2              Books           62.74    item.dat
3          3              Books           98.67    item.dat
4          4              Books           40.95    item.dat


### Enregistrement des ventes nettoyées dans la zone Silver

In [35]:
dossier_silver_ventes = projet / "silver" / "sales"
dossier_silver_ventes.mkdir(parents=True, exist_ok=True)
df_ventes_silver = df_ventes_clean.withColumn(
    "source_file",
    F.lit("store_sales.dat")
)

ventes_pandas = df_ventes_silver.toPandas()
chemin_silver_ventes = dossier_silver_ventes / "sales.parquet"

ventes_pandas.to_parquet(chemin_silver_ventes,engine="pyarrow",index=False)
print("Fichier créé :", chemin_silver_ventes.exists())
print("Nombre de lignes :", len(ventes_pandas))
print("Nombre de colonnes :", len(ventes_pandas.columns))
print("Format :", chemin_silver_ventes.suffix)

Fichier créé : True
Nombre de lignes : 667579
Nombre de colonnes : 24
Format : .parquet


### Vérification de la lecture du fichier Parquet des ventes Silver

In [36]:
import pandas as pd
ventes_verification = pd.read_parquet(chemin_silver_ventes,engine="pyarrow")
print("Nombre de lignes :", len(ventes_verification))
print("Nombre de colonnes :", len(ventes_verification.columns))


print(ventes_verification[["ss_item_sk","ss_quantity","ss_net_paid","source_file"]].head())

Nombre de lignes : 667579
Nombre de colonnes : 24
   ss_item_sk  ss_quantity ss_net_paid      source_file
0       10001           31      715.48  store_sales.dat
1       10000           54     4500.36  store_sales.dat
2       15556           39     2629.55  store_sales.dat
3        4597           20     2668.00  store_sales.dat
4        3258           76      900.60  store_sales.dat


### Enregistrement des ventes agrégées dans la zone Gold

In [37]:
dossier_gold = projet / "gold" / "sales_by_category"
dossier_gold.mkdir(parents=True, exist_ok=True)
categories_pandas = df_ventes_categories.toPandas()
chemin_gold = dossier_gold / "sales_by_category.parquet"
categories_pandas.to_parquet(chemin_gold,engine="pyarrow",index=False)
print("Fichier créé :", chemin_gold.exists())
print("Nombre de catégories :", len(categories_pandas))
print("Nombre de colonnes :", len(categories_pandas.columns))
print("Format :", chemin_gold.suffix)

Fichier créé : True
Nombre de catégories : 9
Nombre de colonnes : 4
Format : .parquet


### Vérification du fichier Parquet de la zone Gold

In [38]:
import pandas as pd
gold_verification = pd.read_parquet(chemin_gold,engine="pyarrow")
print("Nombre de catégories :", len(gold_verification))
print("Nombre de colonnes :", len(gold_verification.columns))
print("\n=== DONNÉES GOLD ===")
print(gold_verification.to_string(index=False))

Nombre de catégories : 9
Nombre de colonnes : 4

=== DONNÉES GOLD ===
              i_category  nombre_lignes_vente  quantite_totale montant_total
                   Music               301236         15238374  519666693.84
                   Books               177671          9007511  307151503.25
             Movies & TV                43376          2189279   74735891.89
  Clothing & Accessories                32372          1640550   56754855.82
             Electronics                32619          1632098   55715087.49
          Home & Kitchen                24380          1226839   41260511.11
       Sports & Outdoors                23911          1209305   40926938.33
            Toys & Games                16121           809999   27437825.33
Tools & Home Improvement                15893           801519   26880279.31


## Étape 9 : Rapport de qualité des données



### Synthèse des contrôles réalisés

In [39]:
import pandas as pd
controles = [
    ["Volume des ventes", "667579 lignes attendues", 667579, "CONFORME"],
    ["Volume des articles", "17820 lignes attendues", 17820, "CONFORME"],
    ["Unicité des articles", "Aucun identifiant dupliqué", 0, "CONFORME"],
    ["Doublons complets", "Aucune ligne dupliquée", 0, "CONFORME"],
    ["Quantités vendues", "Strictement positives", 0, "CONFORME"],
    ["Prix des articles", "Strictement positifs", 0, "CONFORME"],
    ["Intégrité référentielle", "Tous les articles retrouvés", 0, "CONFORME"],
    ["Valeurs manquantes ventes", "1139 valeurs identifiées", 1139, "À SIGNALER"],
    ["Valeurs manquantes articles", "21203 valeurs identifiées", 21203, "À SIGNALER"],
    ["Conversion des types", "Aucun nouveau NULL", 0, "CONFORME"]
]

df_qualite = pd.DataFrame(
    controles,
    columns=["Controle", "Regle", "Resultat", "Statut"]
)
print("=== RAPPORT DE QUALITÉ DES DONNÉES ===")
print(df_qualite.to_string(index=False))

=== RAPPORT DE QUALITÉ DES DONNÉES ===
                   Controle                       Regle  Resultat     Statut
          Volume des ventes     667579 lignes attendues    667579   CONFORME
        Volume des articles      17820 lignes attendues     17820   CONFORME
       Unicité des articles  Aucun identifiant dupliqué         0   CONFORME
          Doublons complets      Aucune ligne dupliquée         0   CONFORME
          Quantités vendues       Strictement positives         0   CONFORME
          Prix des articles        Strictement positifs         0   CONFORME
    Intégrité référentielle Tous les articles retrouvés         0   CONFORME
  Valeurs manquantes ventes    1139 valeurs identifiées      1139 À SIGNALER
Valeurs manquantes articles   21203 valeurs identifiées     21203 À SIGNALER
       Conversion des types          Aucun nouveau NULL         0   CONFORME


### Enregistrement du rapport de qualité des données

In [40]:
chemin_rapport = dossier_output / "rapport_qualite.csv"
# Enregistrer le tableau au format CSV
df_qualite.to_csv(chemin_rapport,sep=";",index=False,encoding="utf-8-sig")
print("Rapport créé :", chemin_rapport.exists())
print("Nombre de contrôles :", len(df_qualite))
print("Chemin :", chemin_rapport)

Rapport créé : True
Nombre de contrôles : 10
Chemin : C:\Users\houda\NoSQL, transformations et Data Lakehouse\output\rapport_qualite.csv


### Comparaison de la taille des fichiers CSV et Parquet

In [41]:
chemin_csv = dossier_output / "products_comparaison.csv"

articles_pandas.to_csv(chemin_csv,index=False,encoding="utf-8")
taille_csv = chemin_csv.stat().st_size / (1024 * 1024)
taille_parquet = chemin_silver_articles.stat().st_size / (1024 * 1024)
gain = (1 - taille_parquet / taille_csv) * 100
print("=== COMPARAISON CSV / PARQUET ===")
print(f"Taille CSV : {taille_csv:.2f} Mo")
print(f"Taille Parquet : {taille_parquet:.2f} Mo")
print(f"Gain d'espace : {gain:.2f} %")

=== COMPARAISON CSV / PARQUET ===
Taille CSV : 6.66 Mo
Taille Parquet : 3.60 Mo
Gain d'espace : 45.91 %


### Comparaison des temps de lecture CSV et Parquet

In [42]:
import time
import pandas as pd

debut_csv = time.perf_counter()
df_csv_test = pd.read_csv(chemin_csv)
temps_csv = time.perf_counter() - debut_csv
debut_parquet = time.perf_counter()

df_parquet_test = pd.read_parquet(chemin_silver_articles,engine="pyarrow")
temps_parquet = time.perf_counter() - debut_parquet

print("=== TEMPS DE LECTURE ===")
print(f"CSV : {temps_csv:.4f} secondes")
print(f"Parquet : {temps_parquet:.4f} secondes")
print("\nLignes CSV :", len(df_csv_test))
print("Lignes Parquet :", len(df_parquet_test))

=== TEMPS DE LECTURE ===
CSV : 0.2221 secondes
Parquet : 0.1139 secondes

Lignes CSV : 17820
Lignes Parquet : 17820


## Étape 10 : Organisation et reproductibilité des traitements


### Centralisation des paramètres et des chemins

In [43]:
from pathlib import Path

PROJET = Path.home() / "NoSQL, transformations et Data Lakehouse"
SOURCE_VENTES = PROJET / "data/retail_project/data/store_sales/store_sales.dat"
SOURCE_ARTICLES = PROJET / "data/retail_project/data/item/item.dat"
SILVER_VENTES = PROJET / "silver/sales/sales.parquet"
SILVER_ARTICLES = PROJET / "silver/products/products.parquet"
GOLD_CATEGORIES = PROJET / "gold/sales_by_category/sales_by_category.parquet"
RAPPORT_QUALITE = PROJET / "output/rapport_qualite.csv"
SEPARATEUR = "|"
ENTETE = False

print("Paramètres et chemins du projet définis.")

Paramètres et chemins du projet définis.


### Vérification des fichiers d'entrée et des livrables

In [44]:
# Vérifier la présence des fichiers du projet

chemins_a_verifier = {"Source ventes": SOURCE_VENTES,"Source articles": SOURCE_ARTICLES,"Silver ventes": SILVER_VENTES,
    "Silver articles": SILVER_ARTICLES,"Gold catégories": GOLD_CATEGORIES,"Rapport qualité": RAPPORT_QUALITE
}
for nom, chemin in chemins_a_verifier.items():
    if chemin.is_file():
        print(f"{nom} : OK")
    else:
        print(f"{nom} : FICHIER MANQUANT")

Source ventes : OK
Source articles : OK
Silver ventes : OK
Silver articles : OK
Gold catégories : OK
Rapport qualité : OK


### Validation finale des livrables Silver et Gold

In [45]:
import pyarrow.parquet as pq

# Nombre de lignes attendu pour chaque fichier
livrables = {
    "Silver ventes": (SILVER_VENTES, 667579),
    "Silver articles": (SILVER_ARTICLES, 17820),
    "Gold catégories": (GOLD_CATEGORIES, 9)}
for nom, (chemin, attendu) in livrables.items():

    fichier = pq.ParquetFile(chemin)
    nombre_lignes = fichier.metadata.num_rows

    statut = "OK" if nombre_lignes == attendu else "ERREUR"

    print(f"{nom} : {nombre_lignes} lignes - {statut}")

Silver ventes : 667579 lignes - OK
Silver articles : 17820 lignes - OK
Gold catégories : 9 lignes - OK


## Publication du TP3 sur GitHub



In [57]:
from pathlib import Path

dossier = Path.home()

print("=== NOTEBOOKS DANS TON DOSSIER UTILISATEUR ===")

for fichier in dossier.glob("*.ipynb"):
    print(fichier.name)

=== NOTEBOOKS DANS TON DOSSIER UTILISATEUR ===
MLPRO.ipynb
NoSQL, transformations et Data Lakehouse.ipynb
PROJET.ipynb
projett.ipynb
Untitled.ipynb
Untitled1.ipynb
Untitled2.ipynb
Untitled3.ipynb
Untitled4.ipynb
Untitled5.ipynb
Untitled6.ipynb
Untitled7.ipynb
Untitled8.ipynb
Untitled9.ipynb
